![car interior](car.jpeg)

# Car-ing is Sharing — Multi-Task NLP Chatbot Prototype

A prototype chatbot for an auto dealership, using pre-trained Hugging Face models to handle four distinct NLP tasks on customer car reviews: sentiment classification, English→Spanish translation, extractive QA, and summarization with bias analysis.

In [ ]:
import pandas as pd
import torch
from transformers import logging
logging.set_verbosity(logging.WARNING)

In [ ]:
df = pd.read_csv("data/car_reviews.csv", sep=";")
reviews = df["Review"].tolist()
real_labels = df["Class"].tolist()

## Task 1: Sentiment Classification

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

predicted_labels = classifier(reviews)

label_map = {"POSITIVE": 1, "NEGATIVE": 0}
predictions = [label_map[pred["label"]] for pred in predicted_labels]
true_labels = [label_map[label] for label in real_labels]

accuracy_result = accuracy_score(true_labels, predictions)
f1_result = f1_score(true_labels, predictions)

print(f"Predictions: {predictions}")
print(f"Accuracy: {accuracy_result:.4f}")
print(f"F1 score: {f1_result:.4f}")

## Task 2: English → Spanish Translation + BLEU Score

In [ ]:
import evaluate

first_review = reviews[0]

sentences = [s.strip() for s in first_review.split(".") if s.strip()]
first_two_sentences = ". ".join(sentences[:2]) + "."

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")
translation_output = translator(first_two_sentences)
translated_review = translation_output[0]["translation_text"]

with open("data/reference_translations.txt", "r", encoding="utf-8") as f:
    references = [line.strip() for line in f if line.strip()]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=[translated_review],
    references=[references]
)

print(f"Translated review: {translated_review}")
print(f"BLEU score: {bleu_score}")

## Task 3: Extractive QA

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

context = reviews[1]
question = "What did he like about the brand?"

model_name = "deepset/minilm-uncased-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(model_name)

inputs = qa_tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = qa_model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1

answer_tokens = inputs["input_ids"][0][start_idx:end_idx]
answer = qa_tokenizer.decode(answer_tokens, skip_special_tokens=True)

print(f"Answer: {answer}")

## Task 4: Summarization + Bias Analysis

In [ ]:
last_review = reviews[-1]

summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
summary_output = summarizer(last_review, max_length=55, min_length=50, do_sample=False)
summarized_text = summary_output[0]["summary_text"]

print(f"Summary: {summarized_text}")

toxicity_metric = evaluate.load("toxicity")
toxicity_results = toxicity_metric.compute(predictions=[summarized_text])
max_toxicity = max(toxicity_results["toxicity"])

print(f"Max toxicity score: {max_toxicity:.4f}")

regard_metric = evaluate.load("regard")
regard_results = regard_metric.compute(data=[summarized_text])

print(f"Regard scores: {regard_results}")

## What I'd improve next
- Test on a larger review sample — 5 reviews isn't enough to draw reliable conclusions about accuracy or bias
- Try a multilingual model to compare translation quality against Helsinki-NLP's dedicated EN→ES model
- Add a lightweight interface (Gradio/Streamlit) so this is an actual interactive demo, not just a notebook

## Note
Car review dataset and reference translations were provided as part of a training exercise.